## **3D VISION AND EXTENDED REALITY**

# Project №7 - 3DGS OF FACES WITH AUGMENTATION

> The description of the project: Starting from a single 2D image, generate multiple views of that image via generative editing (es. NanoBanana, PhotoMaker). Via a 3DGS repo [ https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/]( https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) a 3D model of the face. Generate new images and views, verify if they match with the original one.

Master Degree in  ICT for Internet and Multimedia​

by **Iacobet Ecaterina**

### 1. Installation of dependencies and setting up the environment

In [ ]:
!apt-get update -qq
!apt-get install -y -qq colmap sqlite3 xvfb

!pip uninstall -y opencv-python opencv-python-headless -q
!pip install -q opencv-python-headless Pillow numpy

!pip install -q plyfile tqdm insightface onnxruntime

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:

### 2. Preprocessing of pictures generated by NanoBanana

In [ ]:
import os
import shutil
import subprocess
from PIL import Image
from google.colab import files

BASE_DIR = "/content/face_project"
IMAGES_DIR = f"{BASE_DIR}/images"
os.makedirs(IMAGES_DIR, exist_ok=True)

os.chdir(IMAGES_DIR)
uploaded = files.upload()
os.chdir(BASE_DIR)

print("Uploaded successful")

Saving 75L.jpg to 75L.jpg
Saving 30R.jpg to 30R.jpg
Saving 0.jpg to 0.jpg
Saving 75R.jpg to 75R.jpg
Saving 60L.jpg to 60L.jpg
Saving 60R.jpg to 60R.jpg
Saving 30L.jpg to 30L.jpg
Saving 15R.jpg to 15R.jpg
Saving 15L.jpg to 15L.jpg
Saving 90R.jpg to 90R.jpg
Saving 90L.jpg to 90L.jpg
Saving 45R.jpg to 45R.jpg
Saving 45L.jpg to 45L.jpg
Uploaded successful


### 3. COLMAP used for camer'a poses estimation
Link - https://github.com/colmap/colmap

In [ ]:
SPARSE_DIR = f"{BASE_DIR}/sparse"
DATABASE   = f"{BASE_DIR}/database.db"

if os.path.exists(DATABASE):
    os.remove(DATABASE)
shutil.rmtree(SPARSE_DIR, ignore_errors=True)
os.makedirs(SPARSE_DIR, exist_ok=True)

In [ ]:
!xvfb-run -a colmap feature_extractor \
    --database_path {DATABASE} \
    --image_path {IMAGES_DIR} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.max_image_size 1200 \
    --SiftExtraction.max_num_features 6000 \
    --SiftExtraction.estimate_affine_shape 1 \
    --SiftExtraction.domain_size_pooling 1

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'

Feature extraction

Processed file [1/13]
  Name:            0.jpg
  Dimensions:      1024 x 1024
  Camera:          #1 - PINHOLE
  Focal Length:    1228.80px
  Features:        1661
Processed file [2/13]
  Name:            15L.jpg
  Dimensions:      1024 x 1024
  Camera:          #1 - PINHOLE
  Focal Length:    1228.80px
  Features:        1835
Processed file [3/13]
  Name:            15R.jpg
  Dimensions:      1024 x 1024
  Camera:          #1 - PINHOLE
  Focal Length:    1228.80px
  Features:        1791
Processed file [4/13]
  Name:            30L.jpg
  Dimensions:      1024 x 1024
  Camera:          #1 - PINHOLE
  Focal Length:    1228.80px
  Features:        1979
Processed file [5/13]
  Name:            30R.jpg
  Dimensions:      1024 x 1024
  Camera:          #1 - PINHOLE
  Focal Length:    1228.80px
  Features:        1600
Processed file [6/13]
  Name:            45L.jpg
  Dimensions:      1024 x 1024
 

In [ ]:
!xvfb-run -a colmap exhaustive_matcher \
    --database_path {DATABASE} \
    --SiftMatching.max_num_matches 50000 \
    --SiftMatching.guided_matching 1

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'

Exhaustive feature matching

Matching block [1/1, 1/1] in 53.774s
Elapsed time: 0.898 [minutes]


In [ ]:
!xvfb-run -a colmap mapper \
    --database_path {DATABASE} \
    --image_path {IMAGES_DIR} \
    --output_path {SPARSE_DIR} \
    --Mapper.min_num_matches 10 \
    --Mapper.init_min_num_inliers 8 \
    --Mapper.init_max_error 4


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 30 in 0.000s
Loading images... 13 in 0.002s (connected 13)
Building correspondence graph... in 0.001s (ignored 0)

Elapsed time: 0.000 [minutes]


Finding good initial image pair


Initializing with image pair #2 and #8


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  8.238127e+02    0.00e+00    2.57e+04   0.00e+00   0.00e+00  1.00e+04        0    2.64e-04    7.98e-04
   1  6.802292e+02    1.44e+02    6.41e+04   1.01e+02   4.00e-01  9.92e+03        1    4.38e-04    1.27e-03
   2  4.944567e+02    1.86e+02    3.85e+03   6.22e+01   8.47e-01  1.49e+04        1    3.43e-04    1.64e-03
   3  6.052238e+02   -1.11e+02    3.85e+03   9.52e+01  -2.72e+00  7.44e+03        1    1.73e-04    1.85e-03
   4  4.669876e+02    2.75e+01    3.43e+03   4.88e+01   7.32e-01  8.25e+03        1    2.78e-04    2.15e-03
   5  4.656419e+02    1.35e+00    3

In [ ]:
recon_folders = sorted(f for f in os.listdir(SPARSE_DIR) if f.isdigit())

if not recon_folders:
    raise RuntimeError("COLMAP failed")

RECON_PATH = os.path.join(SPARSE_DIR, recon_folders[0])
print(f"Reconstruction: {RECON_PATH}")

!xvfb-run -a colmap model_analyzer --path {RECON_PATH}

Reconstruction: /content/face_project/sparse/0
Cameras: 1
Images: 13
Registered images: 13
Points: 677
Observations: 1797
Mean track length: 2.654357
Mean observations per image: 138.230769
Mean reprojection error: 1.182041px


In [ ]:
MODEL_PLY = f"{BASE_DIR}/model.ply"

!xvfb-run -a colmap model_converter \
    --input_path {RECON_PATH} \
    --output_path {MODEL_PLY} \
    --output_type PLY

if not os.path.exists(MODEL_PLY):
    raise RuntimeError("PLY conversion failed")

print(f"Saved: {MODEL_PLY}")

Saved: /content/face_project/model.ply


### 4. Import github repository and install dependencies for 3DGS

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git

repo_path = "/content/gaussian-splatting"
%cd {repo_path}

!nvidia-smi
!python -c "import torch; print(torch.__version__, torch.version.cuda)"

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.72 MiB | 18.69 MiB/s, done.
Resolving deltas: 100% (593/593), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects:

In [ ]:
!pip install ninja -q
!pip install submodules/diff-gaussian-rasterization -q --no-build-isolation
!pip install submodules/simple-knn -q --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


### 5. 3DGS

In [ ]:
ITERATIONS = 5000

%cd {repo_path}

!python train.py \
    -s {BASE_DIR} \
    -m {BASE_DIR}/output_3dgs \
    --iterations {ITERATIONS} \
    --save_iterations {ITERATIONS} \
    --test_iterations {ITERATIONS} \
    --eval

/content/gaussian-splatting
2026-09-05 17:59:46.357426: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/face_project/output_3dgs
Output folder: /content/face_project/output_3dgs [05/09 17:59:50]
------------LLFF HOLD------------- [05/09 17:59:50]
Reading camera 13/13 [05/09 17:59:50]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [05/09 17:59:50]
Loading Training Cameras [05/09 17:59:50]
Loading Test Cameras [05/09 17:59:51]
Number of points at initialisation :  677 [05/09 17:59:51]
Training progress: 100% 5000/5000 [09:09<00:00,  9.10it/s, Loss=0.0277278, Depth Loss=0.0000000]

[ITER 5000] Evaluating test: L1 0.07459041848778725 PSNR 17.767029762268066 [05/09 18:09:08]

[ITER 5000] Evaluat

### 5. Rendering

In [ ]:
model_path = f"{BASE_DIR}/output_3dgs"

%cd {repo_path}
!python render.py -m {model_path}
!python metrics.py -m {model_path}

/content/gaussian-splatting
Looking for config file in /content/face_project/output_3dgs/cfg_args
Config file found: /content/face_project/output_3dgs/cfg_args
Rendering /content/face_project/output_3dgs
Loading trained model at iteration 5000 [05/09 18:09:19]
------------LLFF HOLD------------- [05/09 18:09:19]
Reading camera 13/13 [05/09 18:09:19]
Loading Training Cameras [05/09 18:09:19]
Loading Test Cameras [05/09 18:09:20]
Rendering progress: 100% 11/11 [00:06<00:00,  1.81it/s]
Rendering progress: 100% 2/2 [00:01<00:00,  1.91it/s]

Scene: /content/face_project/output_3dgs
Method: ours_5000
Metric evaluation progress:   0% 0/2 [00:00<?, ?it/s]Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth

  0% 0.00/528M [00:00<?, ?B/s]
  2% 10.4M/528M [00:00<00:05, 101MB/s]
  6% 29.6M/528M [00:00<00:03, 158MB/s]
  9% 47.9M/528M [00:00<00:02, 173MB/s]
 12% 64.5M/528M [00:00<00:02, 167MB/s]
 15% 80.6M/528M [00:00<00:03, 1

Loading scene - cameras and dataset

In [ ]:
import sys, os
sys.path.append("/content/gaussian-splatting")
import torch, numpy as np
from PIL import Image
from argparse import Namespace
from scipy.spatial.transform import Rotation as Rot, Slerp
from scene import Scene, GaussianModel
from scene.cameras import MiniCam
from gaussian_renderer import render as gs_render
from utils.graphics_utils import getWorld2View2, getProjectionMatrix

gaussians = GaussianModel(sh_degree=3)

scene_args = Namespace(
    source_path=BASE_DIR,
    model_path=model_path,
    images="images",
    resolution=-1,
    white_background=True,
    data_device="cuda",
    eval=False,
    sh_degree=3,
    depths="",
    train_test_exp=False,
)

scene = Scene(scene_args, gaussians, load_iteration=None, shuffle=False)

Reading camera 13/13
Loading Training Cameras
Loading Test Cameras
Number of points at initialisation :  677


Loading the trained 3D model

In [ ]:
ply_path = os.path.join(model_path, "point_cloud", f"iteration_5000", "point_cloud.ply")
gaussians.load_ply(ply_path)

print("Number of points:", gaussians.get_xyz.shape[0])

Number of points: 61892


Taking parameters from a reference camera and rendering

In [ ]:
ref_cam = scene.getTrainCameras()[0]

width, height = ref_cam.image_width, ref_cam.image_height
FoVx, FoVy = ref_cam.FoVx, ref_cam.FoVy
znear, zfar = 0.01, 100.0

pipe = Namespace(
    convert_SHs_python=False,
    compute_cov3D_python=False,
    debug=False,
    antialiasing=False,
)
bg_color = torch.tensor([1.0, 1.0, 1.0], dtype=torch.float32, device="cuda")

xyz = gaussians.get_xyz.detach().cpu().numpy()
center = np.median(xyz, axis=0)

all_cameras = scene.getTrainCameras() + scene.getTestCameras()

Camera interpolation

In [ ]:
def interpolate_cameras(cam_a, cam_b, t):
    Ra, Rb = cam_a.R.astype(np.float64), cam_b.R.astype(np.float64)
    Ca = -Ra @ cam_a.T.astype(np.float64)
    Cb = -Rb @ cam_b.T.astype(np.float64)

    R_t = Slerp([0, 1], Rot.from_matrix([Ra, Rb]))([t]).as_matrix()[0]
    C_t = (1 - t) * Ca + t * Cb

    R_new = R_t.astype(np.float32)
    T_new = (-R_new.T @ C_t).astype(np.float32)
    return R_new, T_new

Render an image from a given camera pose

In [ ]:
def render_from_RT(R, T):
    world_view = torch.tensor(getWorld2View2(R, T)).transpose(0, 1).cuda()
    proj = getProjectionMatrix(znear=znear, zfar=zfar, fovX=FoVx, fovY=FoVy).transpose(0, 1).cuda()
    full_proj = world_view.unsqueeze(0).bmm(proj.unsqueeze(0)).squeeze(0)

    cam = MiniCam(width=width, height=height, fovy=FoVy, fovx=FoVx,
                  znear=znear, zfar=zfar,
                  world_view_transform=world_view, full_proj_transform=full_proj)

    with torch.no_grad():
        rendering = gs_render(cam, gaussians, pipe, bg_color)["render"]

    arr = rendering.clamp(0, 1).mul(255).byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(arr)

Computing camera azimuth angle around the face center

In [ ]:
def azimuth(cam):
    C = -cam.R @ cam.T
    v = C - center
    return np.degrees(np.arctan2(v[0], v[2]))

sorted_cams = sorted(all_cameras, key=azimuth)

Frame rendering

In [ ]:
rendering = os.path.join(os.path.dirname(model_path), "rendering")
os.makedirs(rendering, exist_ok=True)

steps_between = 3
count = 0

for i in range(len(sorted_cams) - 1):
    for s in range(1, steps_between + 1):
        t = s / (steps_between + 1)
        R, T = interpolate_cameras(sorted_cams[i], sorted_cams[i + 1], t)
        img = render_from_RT(R, T)
        img.save(f"{rendering}/frame_{count}.png")
        count += 1

print(f"{count} frames saved to {rendering}")

36 frames saved to /content/face_project/rendering


### 5. Face matching
Link to InsightFace - https://github.com/deepinsight/insightface

In [ ]:
original_photo_path = "/content/picture0.png"

In [ ]:
import cv2
from insightface.app import FaceAnalysis

face_app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
face_app.prepare(ctx_id=-1, det_size=(640, 640))

def get_embedding(img_path: str):
    img = cv2.imread(img_path)
    faces = face_app.get(img)
    if not faces:
        return None
    faces = sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    return faces[-1].normed_embedding

orig_emb = get_embedding(original_photo_path)
assert orig_emb is not None, "No face detected"

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 91946.72KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [ ]:
import glob

THRESHOLD = 0.45
renders_dir = rendering

results = []
for p in sorted(glob.glob(f"{renders_dir}/*.png")):
    emb = get_embedding(p)
    name = os.path.basename(p)
    if emb is None:
        print(f"{name}: no face detected")
        results.append(None)
        continue
    sim = float(np.dot(orig_emb, emb))
    match = sim > THRESHOLD
    print(f"{name}: cosine_similarity={sim}  match={match}")
    results.append(sim)

valid = [s for s in results if s is not None]

print(f"\nDetectable face: {len(valid)}/{len(results)}")
if valid:
    print(f"Mean cosine similarity: {np.mean(valid)}")
    print(f"Min cosine similarity: {np.min(valid)}")
    print(f"Max cosine similarity: {np.max(valid)}")
    print(f"Count of matched: {sum(s > THRESHOLD for s in valid)}/{len(valid)}")

frame_0.png: no face detected
frame_1.png: no face detected
frame_10.png: cosine_similarity=0.6638492345809937  match=True
frame_11.png: cosine_similarity=0.6436766386032104  match=True
frame_12.png: cosine_similarity=0.5763651728630066  match=True
frame_13.png: cosine_similarity=0.5923279523849487  match=True
frame_14.png: cosine_similarity=0.6350368857383728  match=True
frame_15.png: cosine_similarity=0.6445189714431763  match=True
frame_16.png: cosine_similarity=0.5942208170890808  match=True
frame_17.png: cosine_similarity=0.19112932682037354  match=False
frame_18.png: no face detected
frame_19.png: no face detected
frame_2.png: cosine_similarity=0.7478092908859253  match=True
frame_20.png: no face detected
frame_21.png: cosine_similarity=0.45896780490875244  match=True
frame_22.png: cosine_similarity=0.34670290350914  match=False
frame_23.png: cosine_similarity=0.5003039240837097  match=True
frame_24.png: cosine_similarity=0.5869232416152954  match=True
frame_25.png: cosine_simila

**Conclusion**: Most rendered views (30/31) are recognizable as the same person, so the identity is generally preserved. However, there is large gap between Train PSNR and Test PSNR. That means that the model overfits to the training views and may not be geometrically consistent.